🇬🇧 [English](01_eda.ipynb) · 🇪🇸 [Español](01_eda_es.ipynb)

# Cohorte PBC: análisis exploratorio

Esta notebook es una compañera de EDA ejecutable de los módulos de producción en `src/`.
La fuente de verdad es `data/raw/pbc.csv` y los loaders/summarizers mantenidos en `src/`.

Pregunta principal: ¿cómo se relacionan las variables clínicas y de laboratorio basales
con Estadio 3–4 versus Estadio 1–2? Esto es investigación exploratoria, **no uso
clínico**, y no es un reemplazo de biopsia ni una ayuda de decisión diagnóstica. Ver
[guía AASLD de PBC](https://www.aasld.org/practice-guidelines/primary-biliary-cholangitis)
y [guía EASL de PBC](https://easl.eu/publication/management-of-cholestatic-liver-diseases/)
para contexto clínico.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import skew

# Resuelve la raíz del proyecto, sea que se lance desde cirrhosis/ o cirrhosis/notebooks/.
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
PROJECT_ROOT = next(p for p in _candidates if (p / "src" / "data.py").exists() and (p / "data" / "raw" / "pbc.csv").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    EXPECTED_COLUMNS,
    LEAKAGE_COLUMNS,
    TRIAL_COHORT_MAX_ID,
    add_cohort_indicator,
    events_per_variable,
    load_pbc_data,
    make_targets,
    predictor_frame,
)
from src.eda import summarize_dataset


In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "pbc.csv"
frame = load_pbc_data(DATA_PATH)
summary = summarize_dataset(frame)
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Fuente validada cargada: {DATA_PATH}")
print(f"Shape: {frame.shape}; schema_valid={summary['schema_valid']}")


Raíz del proyecto: /home/datakrdo/Documents/portfolio/cirrhosis
Fuente validada cargada: /home/datakrdo/Documents/portfolio/cirrhosis/data/raw/pbc.csv
Shape: (418, 20); schema_valid=True


El loader de producción confirma el shape esperado de 418×20 y el orden exacto de
columnas, así que el análisis queda anclado al CSV restaurado de Mayo PBC en lugar de
transformaciones locales a la notebook.

Esta cohorte de la era de tratamiento 1974–1984 es histórica y seleccionada, así que sus
mediciones describen una cohorte de investigación observacional, no la población
diagnóstica de hoy.

El análisis continúa siendo solo descriptivo: se preserva la fuente y los resultados no
se presentan como guía clínica ni como reemplazo de biopsia.

In [3]:
print("Columnas esperadas:", EXPECTED_COLUMNS)
print("\nDtypes:")
print(frame.dtypes.to_string())
print(f"\nFilas duplicadas: {summary['duplicate_rows']}")
print(f"Filas con Stage etiquetado: {frame['Stage'].notna().sum()}; sin etiquetar: {frame['Stage'].isna().sum()}")


Columnas esperadas: ['ID', 'N_Days', 'Status', 'Drug', 'Age', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin', 'Stage']

Dtypes:
ID                 int64
N_Days             int64
Status               str
Drug                 str
Age                int64
Sex                  str
Ascites              str
Hepatomegaly         str
Spiders              str
Edema                str
Bilirubin        float64
Cholesterol      float64
Albumin          float64
Copper           float64
Alk_Phos         float64
SGOT             float64
Tryglicerides    float64
Platelets        float64
Prothrombin      float64
Stage            float64

Filas duplicadas: 0
Filas con Stage etiquetado: 412; sin etiquetar: 6


Todos los campos esperados están presentes, con 412 valores de Stage etiquetados y 6
filas sin etiquetar; los IDs, el seguimiento (`N_Days`), el status, y Stage son
metadatos/outcomes en lugar de predictores elegibles.

Usar campos de seguimiento u outcome filtraría información posterior que no estaría
disponible en la evaluación basal.

Solo se usan predictores clínicos/de laboratorio basales, las filas sin Stage etiquetado
se descartan para los análisis de target, y las seis filas sin etiquetar se mantienen en
la contabilidad de procedencia.

In [4]:
missing = (frame.isna().sum().rename("missing").to_frame()
           .assign(rate=lambda x: x["missing"] / len(frame))
           .query("missing > 0")
           .sort_values("missing", ascending=False))
print(missing.to_string(formatters={"rate": "{:.1%}".format}))


               missing  rate
Tryglicerides      136 32.5%
Cholesterol        134 32.1%
Copper             108 25.8%
Drug               106 25.4%
Spiders            106 25.4%
Hepatomegaly       106 25.4%
Ascites            106 25.4%
Alk_Phos           106 25.4%
SGOT               106 25.4%
Platelets           11  2.6%
Stage                6  1.4%
Prothrombin          2  0.5%


Los valores faltantes se concentran en triglicéridos (136), colesterol (134), cobre
(108), varios campos categóricos (106 cada uno), y Stage (6) — este no es un dataset de
casos completos.

Los exámenes o tests faltantes pueden reflejar la práctica histórica y las vías de
atención, así que descartar pacientes silenciosamente o reemplazar categorías con la moda
podría distorsionar las comparaciones a lo largo del espectro de la enfermedad.

La ausencia de datos se mantiene explícita para variables categóricas (`__MISSING__`), y
las variables numéricas reciben imputación por mediana ajustada por fold a través del
pipeline de preprocesamiento de producción.

In [5]:
cohort_frame = add_cohort_indicator(frame)
block = ["Drug", "Ascites", "Hepatomegaly", "Spiders", "Alk_Phos", "SGOT", "Copper",
         "Cholesterol", "Tryglicerides"]
missing_by_cohort = cohort_frame.groupby("trial_cohort", observed=True)[block].apply(
    lambda g: g.isna().mean()
)
print(f"Split de trial cohort: randomised = ID <= {TRIAL_COHORT_MAX_ID}, registry = ID > {TRIAL_COHORT_MAX_ID}\n")
print("Tasa de ausencia de datos por cohorte (el mismo bloque de 8 columnas, todas a la vez):")
print((missing_by_cohort * 100).round(1).to_string())
print(f"\nTamaño de la cohorte registry: {(cohort_frame['trial_cohort'] == 'registry').sum()} "
      f"(coincide exactamente con las {frame['Drug'].isna().sum()} filas sin Drug)")


Split de trial cohort: randomised = ID <= 312, registry = ID > 312

Tasa de ausencia de datos por cohorte (el mismo bloque de 8 columnas, todas a la vez):
               Drug  Ascites  Hepatomegaly  Spiders  Alk_Phos   SGOT  Copper  Cholesterol  Tryglicerides
trial_cohort                                                                                            
randomised      0.0      0.0           0.0      0.0       0.0    0.0     0.6          9.0            9.6
registry      100.0    100.0         100.0    100.0     100.0  100.0   100.0        100.0          100.0

Tamaño de la cohorte registry: 106 (coincide exactamente con las 106 filas sin Drug)


Las 106 filas sin `Drug` son exactamente la cohorte registry (`ID > 312`), y ese mismo
grupo carece de *todo* el bloque de laboratorio/clínico en ~100% — esto no es ausencia de
datos dispersa MCAR/MAR, son dos subcohortes clínicamente distintas apiladas en un mismo
archivo.

El ensayo PBC de Mayo aleatorizó a 312 pacientes a D-penicilamina/placebo y siguió por
separado a ~106 pacientes elegibles pero no aleatorizados en un registro, sin el
protocolo de evaluación completo del ensayo. `Drug` es por lo tanto un proxy casi
perfecto de la pertenencia a la cohorte en lugar de un efecto de tratamiento (el ensayo no
encontró ninguno).

`Drug` se descarta de los predictores (`src.data.LEAKAGE_COLUMNS`) y se reemplaza con un
indicador explícito `trial_cohort` (`src.data.add_cohort_indicator`); el split
train/test estratifica sobre endpoint x cohorte para que ambas subcohortes estén
representadas en cada split, y el modelo primario (`HistGradientBoostingClassifier`)
trata "no medido" como una señal nativa en lugar de imputar con la mediana entre dos
poblaciones distintas.

In [6]:
labeled, y_binary, y_stage = make_targets(frame)
target_table = y_stage.value_counts().sort_index().rename_axis("stage").to_frame("n")
target_table["percent"] = target_table["n"] / len(y_stage)
binary_table = y_binary.value_counts().sort_index().rename(index={0:"Stage 1-2", 1:"Stage 3-4"}).to_frame("n")
binary_table["percent"] = binary_table["n"] / len(y_binary)
print("Distribución exacta de Stage:")
print(target_table.to_string(formatters={"percent": "{:.1%}".format}))
print("\nEndpoint binario primario:")
print(binary_table.to_string(formatters={"percent": "{:.1%}".format}))


Distribución exacta de Stage:
         n percent
stage             
1       21    5.1%
2       92   22.3%
3      155   37.6%
4      144   35.0%

Endpoint binario primario:
             n percent
advanced              
Stage 1-2  113   27.4%
Stage 3-4  299   72.6%


La cohorte etiquetada contiene 113 observaciones tempranas (Estadios 1–2; 27.4%) y 299
avanzadas (Estadios 3–4; 72.6%), con Estadio 4 como la clase exacta más grande (144).

La enfermedad avanzada domina esta muestra, así que la accuracy sola premiaría una regla
trivial de estadio avanzado y ocultaría la enfermedad temprana no detectada.

Estadio 3–4 versus 1–2 es el endpoint primario preespecificado; se usan estratificación y
modelos conscientes del desbalance de clases, y se reportan AUROC/AUPRC más sensibilidad
y especificidad en lugar de solo accuracy.

In [7]:
predictors = predictor_frame(labeled)
numeric = predictors.select_dtypes(include="number").columns.tolist()
rows = []
for column in numeric:
    grouped = labeled.groupby("Stage")[column]
    medians = grouped.median()
    q1 = grouped.quantile(0.25)
    q3 = grouped.quantile(0.75)
    for stage in sorted(medians.index):
        rows.append({"feature": column, "stage": int(stage), "median": medians.loc[stage], "IQR": f"{q1.loc[stage]:.2f}–{q3.loc[stage]:.2f}"})
stage_numeric = pd.DataFrame(rows)
print("Predictores numéricos:", numeric)
print(stage_numeric.pivot(index="feature", columns="stage", values="median").round(2).to_string())
print("\nCuantiles numéricos generales:")
print(predictors[numeric].describe(percentiles=[.25,.5,.75]).T[["count","25%","50%","75%"]].round(2).to_string())


Predictores numéricos: ['Age', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin']
stage                 1         2         3         4
feature                                              
Age            16929.00  17897.00  17947.00  19724.00
Albumin            3.77      3.62      3.61      3.34
Alk_Phos         706.00   1164.00   1257.50   1428.00
Bilirubin          0.80      0.95      1.30      2.55
Cholesterol      239.00    298.00    324.00    299.00
Copper            64.00     49.50     67.50     98.50
Platelets        270.50    277.00    252.00    216.00
Prothrombin       10.15     10.40     10.40     11.00
SGOT              64.32    108.50    112.38    122.45
Tryglicerides     84.00    101.00    119.00    106.00

Cuantiles numéricos generales:
               count       25%       50%       75%
Age            412.0  15609.25  18628.00  21200.50
Bilirubin      412.0      0.80      1.40      3.40
Cholesterol    284.0  

Las medianas estratificadas por estadio muestran mayor bilirrubina, cobre, fosfatasa
alcalina, SGOT, y triglicéridos pero menor albúmina y plaquetas en estadios avanzados; los
IQR amplios y los conteos de datos faltantes indican solapamiento e incertidumbre.

Estos patrones de laboratorio son compatibles con colestasis progresiva, disfunción
sintética, y cambios relacionados con hipertensión portal, pero ninguno es lo bastante
específico para estadificar a un individuo sin evaluación clínica.

Las variables numéricas pasan al modelado multivariable con imputación/escalado seguros
por fold; los contrastes univariados aquí se tratan como generadores de hipótesis, no
como umbrales diagnósticos.

In [8]:
# Los labs con sesgo a la derecha distorsionan la imputación por mediana y los
# coeficientes de modelos lineales; log1p es el fix estándar de estabilización de
# varianza para labs clínicos positivos y sesgados a la derecha, y la evidencia propia
# de CV de este proyecto lo confirma (+0.4pp AUROC para el comparador logistic).
skew_table = predictors[numeric].apply(lambda s: skew(s.dropna())).rename("raw_skew").to_frame()
skew_table["log1p_skew"] = predictors[numeric].apply(lambda s: skew(np.log1p(s.dropna())))
skew_table["max_over_median"] = predictors[numeric].apply(lambda s: s.max() / s.median())
print(skew_table.round(2).sort_values("raw_skew", ascending=False).to_string())


               raw_skew  log1p_skew  max_over_median
Cholesterol        3.39        1.19             5.74
Alk_Phos           2.98        0.91            11.01
Bilirubin          2.70        1.14            20.00
Tryglicerides      2.51        0.36             5.54
Copper             2.29       -0.14             8.05
Prothrombin        2.21        1.54             1.70
SGOT               1.44       -0.07             3.99
Platelets          0.43       -0.59             2.26
Age                0.10       -0.33             1.54
Albumin           -0.46       -0.83             1.31


Siete labs (Bilirubin, Cholesterol, Alk_Phos, SGOT, Tryglicerides, Copper, Prothrombin)
tienen sesgo crudo de 1.45–3.41 y ratios max/median de 5–20x, frente a Age, Albumin, y
Platelets casi simétricos; log1p baja el sesgo a |0.4–1.5| para los siete.

Los ensayos de laboratorio como bilirrubina y fosfatasa alcalina son fisiológicamente
sesgados a la derecha — una pequeña fracción de pacientes muy enfermos genera valores
extremos — así que log1p comprime esa cola sin descartarla, a diferencia de winsorizar o
eliminar outliers.

`np.log1p` se aplica exactamente a estas siete columnas dentro del pipeline de
preprocesamiento ajustado por fold (`src.preprocessing.SKEWED_NUMERIC_COLUMNS`), dejando
Age/Albumin/Platelets sin transformar. Esta es una decisión de preprocesamiento validada
por el score de CV held-out, no una cosmética.

In [9]:
# trial_cohort reemplazó a Drug en `predictors` (ver src.data.predictor_frame);
# usar `predictors` directamente mantiene esta tabla cruzada sincronizada con lo que ven
# los modelos.
categorical = predictors.select_dtypes(exclude="number").columns.tolist()
for column in categorical:
    values = predictors[column].astype(object).fillna("__MISSING__")
    table = pd.crosstab(values, y_binary.map({0: "Stage 1-2", 1: "Stage 3-4"}), normalize="columns")
    print(f"\n{column}: proporción avanzada dentro del endpoint")
    print((table * 100).round(1).to_string())



Sex: proporción avanzada dentro del endpoint
advanced  Stage 1-2  Stage 3-4
Sex                           
F              90.3       89.0
M               9.7       11.0

Ascites: proporción avanzada dentro del endpoint
advanced     Stage 1-2  Stage 3-4
Ascites                          
N                 71.7       69.2
Y                  1.8        7.4
__MISSING__       26.5       23.4

Hepatomegaly: proporción avanzada dentro del endpoint
advanced      Stage 1-2  Stage 3-4
Hepatomegaly                      
N                  56.6       29.4
Y                  16.8       47.2
__MISSING__        26.5       23.4

Spiders: proporción avanzada dentro del endpoint
advanced     Stage 1-2  Stage 3-4
Spiders                          
N                 64.6       49.8
Y                  8.8       26.8
__MISSING__       26.5       23.4

Edema: proporción avanzada dentro del endpoint
advanced  Stage 1-2  Stage 3-4
Edema                         
N              93.8       80.9
S               5.3

Los niveles categóricos muestran proporciones no uniformes de estadio avanzado, mientras
que los niveles faltantes están presentes y son informativos sobre la disponibilidad de
la medición; las celdas pequeñas hacen que los porcentajes crudos sean inestables.
`trial_cohort` en sí muestra una asociación con el estadio, consistente con que
representa una vía de atención distinta más que un mecanismo clínico.

Hallazgos como ascitis, hepatomegalia, arañas vasculares, y edema pueden rastrear
hipertensión portal o descompensación, pero la pertenencia a la cohorte
registry/randomised y la disponibilidad de exámenes pueden confundir las asociaciones
crudas.

Los niveles categóricos y la ausencia de datos se preservan en el pipeline de producción
(one-hot para modelos interpretables, dtype categórico nativo para HistGB); no se extraen
conclusiones causales, y el valor predictivo solo se discute después de la evaluación
held-out (nested-CV).

In [10]:
predictors_final = predictor_frame(labeled)
epv = events_per_variable(predictors_final, y_binary)
eda_decisions = {
    "procedencia": "Usar data/raw/pbc.csv validado; solo cohorte histórica de Mayo.",
    "predictores": f"Excluir {LEAKAGE_COLUMNS}; agregar trial_cohort en su lugar.",
    "ausencia_de_datos": "Categorical __MISSING__ / NaN nativo para HistGB; mediana ajustada por fold para otros modelos.",
    "cohorte": "trial_cohort (randomised/registry) estratifica el split; Drug es una columna de fuga, no un predictor.",
    "endpoint": "Primario Stage 3-4 vs Stage 1-2; el estadio exacto es secundario.",
    "tamano_muestra": f"Events-per-variable en la cohorte etiquetada completa = {epv:.2f} (< regla de 10; reportado, no oculto).",
    "resguardo_clinico": "No es reemplazo de biopsia, diagnóstico, tratamiento, ni uso clínico.",
}
print("Resumen conciso de decisiones de EDA:")
for key, decision in eda_decisions.items():
    print(f"- {key}: {decision}")


Resumen conciso de decisiones de EDA:
- procedencia: Usar data/raw/pbc.csv validado; solo cohorte histórica de Mayo.
- predictores: Excluir ['Stage', 'ID', 'N_Days', 'Status', 'Drug']; agregar trial_cohort en su lugar.
- ausencia_de_datos: Categorical __MISSING__ / NaN nativo para HistGB; mediana ajustada por fold para otros modelos.
- cohorte: trial_cohort (randomised/registry) estratifica el split; Drug es una columna de fuga, no un predictor.
- endpoint: Primario Stage 3-4 vs Stage 1-2; el estadio exacto es secundario.
- tamano_muestra: Events-per-variable en la cohorte etiquetada completa = 7.06 (< regla de 10; reportado, no oculto).
- resguardo_clinico: No es reemplazo de biopsia, diagnóstico, tratamiento, ni uso clínico.


El esquema validado, la ausencia sustancial de datos, el endpoint desbalanceado, y los
patrones de laboratorio/clínicos ligados al estadio respaldan un flujo de modelado
exploratorio sin fuga de datos en lugar de un análisis de casos completos o de una sola
variable.

La guía AASLD/EASL y la evaluación del clínico siguen siendo el contexto apropiado; este
dataset histórico, sin validación externa, no puede reemplazar la biopsia u otros
estudios diagnósticos.

El proyecto avanza solo hacia un modelado exploratorio preespecificado y held-out con
incertidumbre y limitaciones transparentes; estos outputs de la notebook no se usan para
la atención del paciente.